# EGen Evaluation Framework Example

This notebook demonstrates how to use the EGen evaluation framework to benchmark models and calculate metrics.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import json
import torch
import matplotlib.pyplot as plt
import numpy as np

from egen.evaluation import run_benchmark, BenchmarkType, calculate_metrics
from egen.model import THL150, ModelConfig

## Load Model

Next, let's load a pre-trained EGen model. For this example, we'll create a small model for demonstration purposes.

In [ ]:
# Define model configuration
config = ModelConfig(
    hidden_size=256,
    num_layers=4,
    num_heads=4,
    intermediate_size=512,
    max_position_embeddings=1024,
    vocab_size=32000,
    domain_routing=True,
    domain_types=['general', 'code', 'math', 'security'],
    domain_layer_allocation={
        'general': [0, 1, 2, 3],
        'code': [0, 1, 2, 3],
        'math': [0, 1, 2, 3],
        'security': [0, 1, 2, 3],
    },
)

# Initialize model
model = THL150(config)

# Check if a checkpoint exists and load it
checkpoint_path = 'path/to/checkpoint.pt'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded checkpoint from {checkpoint_path}')
else:
    print('No checkpoint found, using randomly initialized model')

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()
print(f'Model loaded on {device}')

## Run MMLU Benchmark

Now, let's run the MMLU benchmark on our model.

In [ ]:
# Run MMLU benchmark
mmlu_results = run_benchmark(
    model=model,
    benchmark_type=BenchmarkType.MMLU,
    batch_size=4,
    device=str(device),
    num_samples=10,  # Limit to 10 samples for demonstration
)

print(f"Overall accuracy: {mmlu_results['overall_accuracy']:.4f}")
print(f"Number of examples: {mmlu_results['num_examples']}")

## Visualize Subject Accuracies

Let's visualize the accuracy across different subjects in the MMLU benchmark.

In [ ]:
if 'subject_accuracies' in mmlu_results:
    # Sort subjects by accuracy
    subjects = list(mmlu_results['subject_accuracies'].keys())
    accuracies = list(mmlu_results['subject_accuracies'].values())
    
    # Sort by accuracy
    sorted_indices = np.argsort(accuracies)
    sorted_subjects = [subjects[i] for i in sorted_indices]
    sorted_accuracies = [accuracies[i] for i in sorted_indices]
    
    # Plot
    plt.figure(figsize=(12, 8))
    plt.barh(sorted_subjects, sorted_accuracies, color='skyblue')
    plt.xlabel('Accuracy')
    plt.ylabel('Subject')
    plt.title('MMLU Accuracy by Subject')
    plt.xlim(0, 1)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print('No subject accuracies available')

## Run Multiple Benchmarks

Let's run multiple benchmarks and compare the results.

In [ ]:
# Define benchmarks to run
benchmarks = [
    BenchmarkType.MMLU,
    BenchmarkType.HELLASWAG,
    BenchmarkType.GSM8K,
]

# Run benchmarks
results = {}
for benchmark in benchmarks:
    print(f"Running {benchmark.value} benchmark...")
    results[benchmark.value] = run_benchmark(
        model=model,
        benchmark_type=benchmark,
        batch_size=4,
        device=str(device),
        num_samples=5,  # Limit to 5 samples for demonstration
    )
    print(f"Completed {benchmark.value} benchmark.")

## Compare Benchmark Results

Now, let's compare the results across different benchmarks.

In [ ]:
# Extract overall accuracies
benchmark_names = list(results.keys())
accuracies = [results[name]['overall_accuracy'] for name in benchmark_names]

# Plot comparison
plt.figure(figsize=(10, 6))
plt.bar(benchmark_names, accuracies, color='lightgreen')
plt.xlabel('Benchmark')
plt.ylabel('Accuracy')
plt.title('Benchmark Comparison')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Print results
for name, accuracy in zip(benchmark_names, accuracies):
    print(f"{name}: {accuracy:.4f}")

## Calculate Custom Metrics

Finally, let's demonstrate how to calculate custom metrics for a specific task.

In [ ]:
# Example classification task
predictions = [0, 1, 0, 1, 1, 0, 1, 0]
labels = [0, 1, 1, 1, 0, 0, 1, 1]

# Calculate classification metrics
classification_metrics = calculate_metrics(
    predictions=predictions,
    labels=labels,
    task_type='classification',
)

print("Classification Metrics:")
for metric, value in classification_metrics.items():
    print(f"{metric}: {value:.4f}")

# Example generation task
generated_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "The cat sat on the mat.",
]
reference_texts = [
    "The fast brown fox leaps over the lazy dog.",
    "The feline rested on the carpet.",
]

# Calculate generation metrics (requires rouge and nltk packages)
try:
    generation_metrics = calculate_metrics(
        predictions=generated_texts,
        labels=reference_texts,
        task_type='generation',
        use_rouge=True,
        use_bleu=True,
    )
    
    print("\nGeneration Metrics:")
    for metric, value in generation_metrics.items():
        print(f"{metric}: {value:.4f}")
except ImportError as e:
    print(f"\nCould not calculate generation metrics: {e}")

## Conclusion

In this notebook, we demonstrated how to use the EGen evaluation framework to:

1. Run standard benchmarks like MMLU, HellaSWAG, and GSM8K
2. Visualize benchmark results
3. Calculate and compare metrics across different tasks

This framework provides a standardized way to evaluate EGen models and compare their performance across different tasks and domains.